##Import Required Libraries

In [ ]:
import os
import random
import time
import shutil
import subprocess
import itertools
import re
import traceback
import warnings
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import joblib

from scipy.stats import norm, rankdata, kendalltau
from scipy.optimize import minimize

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_squared_error, r2_score, precision_score, silhouette_score
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import SpectralClustering
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, InputLayer
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.initializers import HeNormal, GlorotUniform
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam, AdamW, Nadam
from tensorflow.keras.losses import sparse_categorical_crossentropy
from tensorflow.keras.metrics import Precision, Recall, AUC, SparseCategoricalAccuracy, Metric
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.backend import clear_session

from networkx.algorithms import bipartite

warnings.filterwarnings('ignore')

# Google Colab specific (only needed if running in Colab)
from google.colab import drive

##Load dataset from Drive

In [ ]:
# Mount your Google Drive
drive.mount('/content/drive')

def processFile(file_path):
  # Define the columns
  columns = ['relevance', 'qid'] + ['F_' + str(i) for i in range(1, 30)] + ['docid']

  # Initialize an empty list to store the rows
  rows = []

  # Open the file and read line by line
  with open(file_path, 'r') as file:
      for line in file:
          # Split the line into parts
          parts = line.strip().split()
          # Extract relevance
          relevance = int(parts[0])

          # Extract qid
          qid = int(parts[1].split(':')[1])

          # Extract features
          features = [float(part.split(':')[1]) for part in parts[2:31]]
          # Extract docid
          docid = int(parts[-1].split(':')[1])

          # Combine all parts into a row
          row = [relevance, qid] + features + [docid]

          # Append the row to the list of rows
          rows.append(row)

    # Create the DataFrame
  df = pd.DataFrame(rows, columns=columns)
  return df

# Specify the path to your dataset.txt file on Google Drive
file_path = '/content/drive/MyDrive/WCL2R/FS.txt'
df_Main = processFile(file_path)

## Split Train Set and Test Set

In [ ]:


def train_test_split2(df, test_size, random_state=None):
    """
    Split DataFrame into train and test sets by qid groups.

    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame with columns including 'qid' and 'docid'
    test_size : float, default=0.29
        Proportion of documents to include in test split for each qid
    random_state : int, default=None
        Random seed for reproducibility

    Returns:
    --------
    df_train : pandas.DataFrame
        Training set containing (1-test_size) percent of documents for each qid
    df_test : pandas.DataFrame
        Test set containing test_size percent of documents for each qid
    """
    if random_state is not None:
        np.random.seed(random_state)

    train_rows = []
    test_rows = []

    # Group by qid
    for qid, group in df.groupby('qid'):
        # Get unique docids for this qid
        docids = group['docid'].unique()

        # Calculate number of test documents for this qid
        n_test = max(1, int(len(docids) * test_size))

        # Randomly select test docids
        test_docids = np.random.choice(docids, size=n_test, replace=False)

        # Split the group into train and test based on selected docids
        test_mask = group['docid'].isin(test_docids)
        test_group = group[test_mask]
        train_group = group[~test_mask]

        # Append to respective lists
        train_rows.append(train_group)
        test_rows.append(test_group)

    # Concatenate all groups
    df_train = pd.concat(train_rows, ignore_index=True)
    df_test = pd.concat(test_rows, ignore_index=True)

    return df_train, df_test



df_train, df_test = train_test_split(df_Main, test_size=0.29, random_state=42)


# Display the shapes of the resulting DataFrames
print("Training set shape:", df_train.shape)
print("Test set shape:", df_test.shape)

# Write df_train to train.txt
df_train.to_csv('train.txt', index=False)

# Write df_test to test.txt
df_test.to_csv('test.txt', index=False)



##Feature Extraction from Graph-based representation of the LTR Dataset

In [ ]:
def weighted_degree_centrality(G, weight='weight'):
    """Calculate weighted degree centrality properly."""
    degree_dict = {}
    for node in G.nodes():
        degree_dict[node] = sum(data.get(weight, 1.0) for _, _, data in G.edges(node, data=True))
    return degree_dict

def bipartite_pagerank(G, alpha=0.85, max_iter=200, tol=1.0e-8, weight='weight'):
    """
    Use NetworkX's optimized PageRank implementation.
    """
    print("Calculating PageRank with NetworkX implementation...")

    try:
        pagerank_dict = nx.pagerank(G, alpha=alpha, max_iter=max_iter, tol=tol, weight=weight)
        return pagerank_dict
    except Exception as e:
        print(f"PageRank failed: {e}, using weighted degree fallback...")
        return weighted_degree_centrality(G, weight=weight)

def bipartite_eigenvector_centrality(G, document_nodes, query_nodes, weight='weight'):
    """
    eigenvector centrality for bipartite graphs using projection method.
    """
    print("Calculating bipartite eigenvector centrality using projection method...")

    try:
        # Method 1: Use projected graphs for eigenvector centrality
        # Project to document-document graph
        document_projection = bipartite.weighted_projected_graph(G, document_nodes)

        # Calculate eigenvector centrality on the projection
        doc_eigenvector = nx.eigenvector_centrality(document_projection, weight=weight, max_iter=1000, tol=1e-6)

        # Project to query-query graph
        query_projection = bipartite.weighted_projected_graph(G, query_nodes)
        query_eigenvector = nx.eigenvector_centrality(query_projection, weight=weight, max_iter=1000, tol=1e-6)

        return doc_eigenvector, query_eigenvector

    except Exception as e:
        print(f"Projection method failed: {e}, using alternative approach...")
        try:
            # Method 2: Use HITS algorithm (more stable for bipartite graphs)
            hubs, authorities = nx.hits(G, max_iter=1000, tol=1e-6, normalized=True)

            # Documents are authorities, queries are hubs in typical bipartite structure
            doc_eigenvector = {node: authorities.get(node, 0) for node in document_nodes}
            query_eigenvector = {node: hubs.get(node, 0) for node in query_nodes}

            return doc_eigenvector, query_eigenvector

        except Exception as e2:
            print(f"HITS failed: {e2}, using weighted degree fallback...")
            # Fallback: weighted degree centrality
            degree_dict = weighted_degree_centrality(G, weight=weight)
            max_degree = max(degree_dict.values()) if degree_dict else 1

            doc_eigenvector = {node: degree_dict.get(node, 0) / max_degree for node in document_nodes}
            query_eigenvector = {node: degree_dict.get(node, 0) / max_degree for node in query_nodes}

            return doc_eigenvector, query_eigenvector

def robust_betweenness_centrality(G, weight='weight'):
    """
    Optimized betweenness centrality calculation.
    """
    print("Calculating betweenness centrality...")

    try:
        k = min(100, len(G) // 20)
        if k < 10:
            k = len(G)

        betweenness_dict = nx.betweenness_centrality(
            G, weight=weight, k=k, normalized=True, endpoints=False, seed=42
        )
        return betweenness_dict

    except Exception as e:
        print(f"Betweenness centrality failed: {e}, using closeness centrality...")
        try:
            closeness_dict = nx.closeness_centrality(G, distance=weight)
            return closeness_dict
        except:
            print("Closeness centrality failed, using weighted degree...")
            return weighted_degree_centrality(G, weight=weight)

def average_neighbor_degree(G, nodes, weight='weight'):
    """Calculate average weighted degree of neighbors for each node."""
    avg_deg_dict = {}
    for node in nodes:
        neighbors = list(G.neighbors(node))
        if not neighbors:
            avg_deg_dict[node] = 0
            continue

        neighbor_degrees = []
        for neighbor in neighbors:
            deg = sum(data.get(weight, 1.0) for _, _, data in G.edges(neighbor, data=True))
            neighbor_degrees.append(deg)

        avg_deg_dict[node] = np.mean(neighbor_degrees) if neighbor_degrees else 0

    return avg_deg_dict

def bipartite_common_neighbors_ratio(G, nodes, bipartite_sets):
    """
    Calculate ratio of neighbor pairs that have at least one common neighbor.
    """
    conn_dict = {}
    query_nodes, doc_nodes = bipartite_sets

    for node in nodes:
        neighbors = list(G.neighbors(node))
        if len(neighbors) < 2:
            conn_dict[node] = 0
            continue

        connected_pairs = 0
        total_pairs = 0

        for i, n1 in enumerate(neighbors):
            for n2 in neighbors[i+1:]:
                total_pairs += 1
                common_neighbors = set(G.neighbors(n1)) & set(G.neighbors(n2))
                common_neighbors.discard(node)

                if common_neighbors:
                    connected_pairs += 1

        conn_dict[node] = connected_pairs / total_pairs if total_pairs > 0 else 0

    return conn_dict

def bipartite_weighted_common_neighbors(G, nodes, bipartite_sets, weight='weight'):
    """
    Calculate weighted common neighbors score for bipartite graphs.
    """
    conn_dict = {}
    query_nodes, doc_nodes = bipartite_sets

    for node in nodes:
        neighbors = list(G.neighbors(node))
        if len(neighbors) < 2:
            conn_dict[node] = 0
            continue

        total_score = 0
        total_pairs = 0

        for i, n1 in enumerate(neighbors):
            for n2 in neighbors[i+1:]:
                total_pairs += 1
                common_neighbors = set(G.neighbors(n1)) & set(G.neighbors(n2))
                common_neighbors.discard(node)

                pair_score = 0
                for cn in common_neighbors:
                    w1 = G[n1][cn].get(weight, 1.0)
                    w2 = G[n2][cn].get(weight, 1.0)
                    pair_score += np.sqrt(w1 * w2)

                total_score += pair_score

        conn_dict[node] = total_score / total_pairs if total_pairs > 0 else 0

    return conn_dict

def create_projection_features(G, document_nodes, query_nodes, weight='weight'):
    """
    Create projection-based features for documents and queries.
    """
    print("Creating document-document projection...")
    try:
        document_projection = bipartite.weighted_projected_graph(G, document_nodes)
        doc_projection_degree = dict(document_projection.degree(weight=weight))
        doc_projection_clustering = nx.clustering(document_projection, weight=weight)
    except Exception as e:
        print(f"Document projection failed: {e}")
        doc_projection_degree = {node: 0 for node in document_nodes}
        doc_projection_clustering = {node: 0 for node in document_nodes}

    print("Creating query-query projection...")
    try:
        query_projection = bipartite.weighted_projected_graph(G, query_nodes)
        query_projection_degree = dict(query_projection.degree(weight=weight))
        query_projection_clustering = nx.clustering(query_projection, weight=weight)
    except Exception as e:
        print(f"Query projection failed: {e}")
        query_projection_degree = {node: 0 for node in query_nodes}
        query_projection_clustering = {node: 0 for node in query_nodes}

    return (doc_projection_degree, doc_projection_clustering,
            query_projection_degree, query_projection_clustering)

def create_bipartite_graph_with_all_features(df):
    """
    eigenvector centrality for bipartite graphs.
    """

    # Create a bipartite graph
    G = nx.Graph()
    G.add_nodes_from(df['qid'], bipartite=0)
    G.add_nodes_from(df['docid'], bipartite=1)

    for _, row in df.iterrows():
        if float(row['relevance']) > 0:
          G.add_edge(row['qid'], row['docid'], weight=float(row['relevance']))

    # Get node sets
    query_nodes = {n for n, d in G.nodes(data=True) if d['bipartite'] == 0}
    document_nodes = {n for n, d in G.nodes(data=True) if d['bipartite'] == 1}
    bipartite_sets = (query_nodes, document_nodes)

    print(f"Graph stats: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print(f"Queries: {len(query_nodes)}, Documents: {len(document_nodes)}")

    # Calculate all centrality measures
    print("\n=== Calculating Centrality Measures ===")

    # PageRank
    pagerank_dict = bipartite_pagerank(G, weight='weight')

    # Eigenvector Centrality for bipartite graphs
    doc_eigenvector_dict, query_eigenvector_dict = bipartite_eigenvector_centrality(
        G, document_nodes, query_nodes, weight='weight'
    )

    # Betweenness Centrality
    betweenness_dict = robust_betweenness_centrality(G, weight='weight')

    # Closeness Centrality
    print("Calculating closeness centrality...")
    try:
        closeness_dict = nx.closeness_centrality(G, distance='weight')
    except:
        print("Closeness centrality failed, using fallback...")
        closeness_dict = weighted_degree_centrality(G, weight='weight')

    # Weighted Degree
    degree_dict = weighted_degree_centrality(G, weight='weight')

    # Calculate other features
    print("\n=== Calculating Additional Features ===")

    # Clustering coefficients
    print("Calculating clustering coefficients...")
    doc_clustering_dict = bipartite.clustering(G, nodes=document_nodes, mode='dot')
    query_clustering_dict = bipartite.clustering(G, nodes=query_nodes, mode='dot')

    # Average Neighbor Degree
    print("Calculating average neighbor degree...")
    doc_avg_neighbor_deg_dict = average_neighbor_degree(G, document_nodes, weight='weight')
    query_avg_neighbor_deg_dict = average_neighbor_degree(G, query_nodes, weight='weight')

    # Common Neighbors Features
    print("Calculating common neighbors features...")
    doc_common_neighbors_ratio = bipartite_common_neighbors_ratio(G, document_nodes, bipartite_sets)
    query_common_neighbors_ratio = bipartite_common_neighbors_ratio(G, query_nodes, bipartite_sets)

    doc_weighted_common_neighbors = bipartite_weighted_common_neighbors(G, document_nodes, bipartite_sets, weight='weight')
    query_weighted_common_neighbors = bipartite_weighted_common_neighbors(G, query_nodes, bipartite_sets, weight='weight')

    # Projection Features
    (doc_projection_degree, doc_projection_clustering,
     query_projection_degree, query_projection_clustering) = create_projection_features(
        G, document_nodes, query_nodes, weight='weight'
    )

    # Prepare all feature dictionaries
    feature_dicts = {
        # Centrality measures
        'doc_pagerank': {node: pagerank_dict[node] for node in document_nodes},
        'query_pagerank': {node: pagerank_dict[node] for node in query_nodes},
        'doc_eigenvector': doc_eigenvector_dict,
        'query_eigenvector': query_eigenvector_dict,
        'doc_betweenness': {node: betweenness_dict[node] for node in document_nodes},
        'query_betweenness': {node: betweenness_dict[node] for node in query_nodes},
        'doc_closeness': {node: closeness_dict[node] for node in document_nodes},
        'query_closeness': {node: closeness_dict[node] for node in query_nodes},
        'doc_degree': {node: degree_dict[node] for node in document_nodes},
        'query_degree': {node: degree_dict[node] for node in query_nodes},

        # Additional features
        'doc_clustering': doc_clustering_dict,
        'query_clustering': query_clustering_dict,
        'doc_avg_neighbor_deg': doc_avg_neighbor_deg_dict,
        'query_avg_neighbor_deg': query_avg_neighbor_deg_dict,
        'doc_common_neighbors_ratio': doc_common_neighbors_ratio,
        'query_common_neighbors_ratio': query_common_neighbors_ratio,
        'doc_weighted_common_neighbors': doc_weighted_common_neighbors,
        'query_weighted_common_neighbors': query_weighted_common_neighbors,
        'doc_projection_degree': doc_projection_degree,
        'doc_projection_clustering': doc_projection_clustering,
        'query_projection_degree': query_projection_degree,
        'query_projection_clustering': query_projection_clustering
    }

    # Add features to DataFrame
    print("\n=== Adding Features to DataFrame ===")
    feature_mappings = [
        ('doc_pagerank', 'docid'), ('query_pagerank', 'qid'),
        ('doc_eigenvector', 'docid'), ('query_eigenvector', 'qid'),
        ('doc_betweenness', 'docid'), ('query_betweenness', 'qid'),
        ('doc_closeness', 'docid'), ('query_closeness', 'qid'),
        ('doc_degree', 'docid'), ('query_degree', 'qid'),
        ('doc_clustering', 'docid'), ('query_clustering', 'qid'),
        ('doc_avg_neighbor_deg', 'docid'), ('query_avg_neighbor_deg', 'qid'),
        ('doc_common_neighbors_ratio', 'docid'), ('query_common_neighbors_ratio', 'qid'),
        ('doc_weighted_common_neighbors', 'docid'), ('query_weighted_common_neighbors', 'qid'),
        ('doc_projection_degree', 'docid'), ('doc_projection_clustering', 'docid'),
        ('query_projection_degree', 'qid'), ('query_projection_clustering', 'qid')
    ]

    for feature_name, id_column in feature_mappings:
        df[feature_name] = df[id_column].map(feature_dicts[feature_name]).fillna(0)
        print(f"Added {feature_name}")

    # Print feature statistics
    print("\n=== Feature Statistics ===")
    graph_features = [col for col in df.columns if col.startswith(('doc_', 'query_')) and col not in ['docid', 'qid']]

    for feature in graph_features:
        non_zero = (df[feature] > 0).sum()
        total = len(df[feature])
        unique_vals = df[feature].nunique()
        print(f"{feature}:")
        print(f"  Range: [{df[feature].min():.6f}, {df[feature].max():.6f}]")
        print(f"  Non-zero: {non_zero}/{total} ({non_zero/total*100:.1f}%)")
        print(f"  Unique values: {unique_vals}")
        print()

    return df
#----------------------------------------------------------------------------------
def train_test_split2(df, test_size , random_state=None):
    """
    Split DataFrame into train and test sets by qid groups.

    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame with columns including 'qid' and 'docid'
    test_size : float, default=0.29
        Proportion of documents to include in test split for each qid
    random_state : int, default=None
        Random seed for reproducibility

    Returns:
    --------
    df_train : pandas.DataFrame
        Training set containing (1-test_size) percent of documents for each qid
    df_test : pandas.DataFrame
        Test set containing test_size percent of documents for each qid
    """
    if random_state is not None:
        np.random.seed(random_state)

    train_rows = []
    test_rows = []

    # Group by qid
    for qid, group in df.groupby('qid'):
        # Get unique docids for this qid
        docids = group['docid'].unique()

        # Calculate number of test documents for this qid
        n_test = max(1, int(len(docids) * test_size))

        # Randomly select test docids
        test_docids = np.random.choice(docids, size=n_test, replace=False)

        # Split the group into train and test based on selected docids
        test_mask = group['docid'].isin(test_docids)
        test_group = group[test_mask]
        train_group = group[~test_mask]

        # Append to respective lists
        train_rows.append(train_group)
        test_rows.append(test_group)

    # Concatenate all groups
    df_train = pd.concat(train_rows, ignore_index=True)
    df_test = pd.concat(test_rows, ignore_index=True)

    return df_train, df_test



source_path = '/content/drive/MyDrive/Eval-Score-4.0-WCL2R.pl'  # Replace with actual path
destination_path = '/content/Eval-Score-4.0-WCL2R.pl'
# Copy the file
!cp {source_path} {destination_path}

test_size =0.29
n_features = 51 #features_list + qid

df_train, df_test = train_test_split2(df_Main, test_size, random_state=42)
#----------------------------------------------------------------------------------
print("Starting graph feature extraction with eigenvector centrality...")
df_train_with_centralities = create_bipartite_graph_with_all_features(df_train)

# Normalization
print("Applying normalization...")
scaler2 = MinMaxScaler()

graph_feature_cols = [col for col in df_train_with_centralities.columns
                    if col.startswith(('doc_', 'query_')) and col not in ['docid', 'qid']]

# Fit and transform the selected columns
df_train_with_centralities[graph_feature_cols] = scaler2.fit_transform(df_train_with_centralities[graph_feature_cols])
#----------------------------------------------------------------------
y_train = df_train_with_centralities[['relevance']].astype(int)

features_df_old = df_train_with_centralities.columns[:]
features_df = features_df_old.drop(['qid'])
features_df = features_df.drop(['docid'])
df_train_withoutqid_docid = df_train_with_centralities[features_df].astype(float)

# Open the output file in write mode
with open(f"/content/judgements.txt", "w") as outfile:
    # Iterate over each row in the DataFrame
    for index, row in df_train_with_centralities.iterrows():
        # Extract the required values
        relevance_value = int(row['relevance'])
        qid_value = int(row['qid'])
        docid_value = int(row['docid'])

        # Format the line as per the requirement
        line = f"{relevance_value} qid:{qid_value} #docid = {docid_value}\n"

        # Write the line to the file
        outfile.write(line)
p_at_1_train = pd.Series(np.zeros(n_features))

for i in range(1, n_features+1):
    predicted_labels = df_train_withoutqid_docid.iloc[:, i]
    with open(f'predictions-F1.txt', 'w') as f:
        for f1 in predicted_labels:
            f.write(f"  {f1:.7e}  \n")

    # Evaluate final predictions using external script
    !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions-F1.txt output.txt 0


    # Read and store evaluation metrics
    df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
    #p_at_1[i].append(float(df.iloc[1, 1]))
    p_at_1_train[i-1] = float(df.iloc[1, 1])


print("Saving results...")
df_train_with_centralities.to_csv(
    'train_with_centralities_22.csv',
    index=False,
    encoding='utf-8'
    ,float_format='%.8f'
)



##Evaluate featues based on their P@1 values

In [ ]:
y_train = df_train_with_centralities[['relevance']].astype(int)
# Count occurrences of each unique value
unique_values, counts = np.unique(y_train, return_counts=True)

# Print results
for value, count in zip(unique_values, counts):
    print(f"Real Number of Train {value}'s: {count}")

features_df_old = df_train_with_centralities.columns[:]
features_df = features_df_old.drop(['qid'])
features_df = features_df.drop(['docid'])
df_train_withoutqid_docid = df_train_with_centralities[features_df].astype(float)


source_path = '/content/drive/MyDrive/Eval-Score-4.0-WCL2R.pl'  # Replace with actual path
destination_path = '/content/Eval-Score-4.0-WCL2R.pl'
# Copy the file
!cp {source_path} {destination_path}

# Open the output file in write mode
with open(f"/content/judgements.txt", "w") as outfile:
    # Iterate over each row in the DataFrame
    for index, row in df_train_with_centralities.iterrows():
        # Extract the required values
        relevance_value = int(row['relevance'])
        qid_value = int(row['qid'])
        docid_value = int(row['docid'])

        # Format the line as per the requirement
        line = f"{relevance_value} qid:{qid_value} #docid = {docid_value}\n"

        # Write the line to the file
        outfile.write(line)

n_features = 51 #features_list + qid
# Initialize lists to store evaluation metrics
p_at_1_train = pd.Series(np.zeros(n_features))

for i in range(1, n_features+1):
    predicted_labels = df_train_withoutqid_docid.iloc[:, i]
    with open(f'predictions-F1.txt', 'w') as f:
        for f1 in predicted_labels:
            f.write(f"  {f1:.7e}  \n")

    # Evaluate final predictions using external script
    !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions-F1.txt output.txt 0


    # Read and store evaluation metrics
    df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
    #p_at_1[i].append(float(df.iloc[1, 1]))
    p_at_1_train[i-1] = float(df.iloc[1, 1])



features_df_0 = features_df.drop(['relevance'])
df_train_withoutqid_docid_rel = df_train_with_centralities[features_df_0].astype(float)

# Create a DataFrame with column names and corresponding p@1 values
result_df = pd.DataFrame({
    'column_name': df_train_withoutqid_docid_rel.columns,
    'p_at_1_value': p_at_1_train.values
})
print(result_df.to_string())

## P@1 Chart

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(result_df['column_name'], result_df['p_at_1_value'], marker='o', linewidth=2, markersize=6)
plt.xlabel('Features')
plt.ylabel('P@1 Value')
plt.title('P@1 Values by Features in WCL2R TrainSet')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=90)  # Rotate x-axis labels if they're long
plt.tight_layout()
plt.show()

##Mean Number of docid per each qids

In [ ]:
# Get the count of docids per qid
docs_per_qid = df_train_with_centralities.groupby('qid')['docid'].nunique()

# Calculate mean
mean_docs = docs_per_qid.mean()

# Also get other statistics
print(f"Mean number of unique docid per qid: {mean_docs:.2f}")
print(f"Median number of unique docid per qid: {docs_per_qid.median():.2f}")
print(f"Standard deviation: {docs_per_qid.std():.2f}")
print(f"Min: {docs_per_qid.min()}, Max: {docs_per_qid.max()}")

# Show the distribution
print("\nDistribution of docid counts per qid:")
print(docs_per_qid.value_counts().sort_index())

In [ ]:

# Method 1: Using groupby with value_counts and normalization
relevance_percentages = df_train_with_centralities.groupby('qid')['relevance'].value_counts(normalize=True).mul(100).reset_index(name='percentage')

# Method 2: More explicit approach
relevance_counts = df_train_with_centralities.groupby(['qid', 'relevance']).size().reset_index(name='count')
docs_per_qid = df_train_with_centralities.groupby('qid')['docid'].nunique().reset_index(name='total_docs')

# Merge and calculate percentages
relevance_percentages = relevance_counts.merge(docs_per_qid, on='qid')
relevance_percentages['percentage'] = (relevance_percentages['count'] / relevance_percentages['total_docs']) * 100

# Display the result
print(relevance_percentages)# Pivot the results for better readability
pivot_table = relevance_percentages.pivot(index='qid', columns='relevance', values='percentage').fillna(0)
print(pivot_table)



In [ ]:

docs_per_qid = df_train_with_centralities.groupby('qid')['docid'].nunique().reset_index()
docs_per_qid.columns = ['qid', 'total_docs']


relevance_counts = df_train_with_centralities.groupby(['qid', 'relevance']).size().reset_index(name='count')


merged_df = relevance_counts.merge(docs_per_qid, on='qid')


all_total_docs = merged_df['total_docs'].unique()
all_relevance_levels = df_train_with_centralities['relevance'].unique()



all_combinations = pd.DataFrame(list(product(all_total_docs, all_relevance_levels)),
                                columns=['total_docs', 'relevance'])


result = merged_df.groupby(['total_docs', 'relevance'])['count'].sum().reset_index()

result_complete = all_combinations.merge(result, on=['total_docs', 'relevance'], how='left')
result_complete['count'] = result_complete['count'].fillna(0)

total_docs_counts = result_complete.groupby('total_docs')['count'].sum().reset_index()
total_docs_counts.columns = ['total_docs', 'total_count']

result_final = result_complete.merge(total_docs_counts, on='total_docs')
result_final['percentage'] = (result_final['count'] / result_final['total_count']) * 100

result_final = result_final.sort_values(['total_docs', 'relevance']).reset_index(drop=True)

print(result_final)
result_final.to_csv("docs_query relavances.csv")

## Calculate Kendall's Tau

In [ ]:

def create_kendall_tau_matrix(df):
  """Creates a matrix of Kendall's Tau coefficients between different features.

  Args:
    df: A pandas DataFrame with columns 'qid', 'feature_1', 'feature_2', ..., 'feature_n', 'docID'.

  Returns:
    A numpy array representing the matrix of Kendall's Tau coefficients.
  """

  #features_df = df.columns[2:]  # Exclude 'qid'
  features_df_old = df.columns[:]
  features_df = features_df_old.drop(['qid'])
  features_df = features_df.drop(['docid'])

  #drop(column=['docid'])
  print(features_df)
  n_features = len(features_df)
  print(n_features)
  kendall_tau_matrix = np.zeros((n_features, n_features))

  for i in range(n_features):
    for j in range(i + 1, n_features):
      feature_i = features_df[i]
      feature_j = features_df[j]

      # Group by 'qid' and calculate Kendall's Tau for each group
      kendall_tau_values = df.groupby('qid').apply(lambda x: kendalltau(x[feature_i], x[feature_j])[0])

      # Calculate the average Kendall's Tau
      avg_kendall_tau = kendall_tau_values.mean()

      # Fill in the matrix symmetrically
      kendall_tau_matrix[i, j] = avg_kendall_tau
      kendall_tau_matrix[j, i] = avg_kendall_tau

  return kendall_tau_matrix

print(df_train_with_centralities.columns)
kendall_tau_matrix = create_kendall_tau_matrix(df_train_with_centralities)

df0 = pd.DataFrame(kendall_tau_matrix)
df0.to_csv('kendall_tau_matrix_22.csv', index=False, header=True)

print(kendall_tau_matrix)


In [ ]:
kendall_tau_matrix = np.nan_to_num(kendall_tau_matrix)

threshold = 0.12  # Adjust this threshold as needed: 0.12
# Set values below the threshold to zero
print(kendall_tau_matrix.shape)
kendall_tau_matrix_without_rel = np.delete(kendall_tau_matrix, 0, axis=0)
kendall_tau_matrix_without_rel = np.delete(kendall_tau_matrix_without_rel, 0, axis=1)
#print(kendall_tau_matrix_without_rel)
kendall_tau_matrix_without_rel[np.abs(kendall_tau_matrix_without_rel) < threshold] = 0

kendall_tau_matrix_without_GF = np.zeros((29, 29))
# Copy the desired portion
kendall_tau_matrix_without_GF[:, :] = kendall_tau_matrix_without_rel[0:29, 0:29]

##Predict Graph-based Features using Basic Features for the TEST dataset

In [ ]:
def find_best_num_clusters(graph_adjacency_matrix, num_clusters_range):

    """
    Automatically finds the best number of clusters for a given weighted graph using spectral clustering.

    Args:
        graph_adjacency_matrix: The adjacency matrix of the graph.
        num_clusters_range: The range of cluster numbers to explore.

    Returns:
        The best number of clusters and the corresponding cluster labels.
    """

    best_num_clusters = None
    best_cluster_labels = None
    best_silhouette_score = -1

    # Ensure reproducibility
    rng = np.random.RandomState(42)

    for num_clusters in num_clusters_range:
        spectral_clustering = SpectralClustering(n_clusters=num_clusters,
                                                affinity='precomputed',
                                                random_state=rng)
        cluster_labels = spectral_clustering.fit_predict(graph_adjacency_matrix)

        silhouette_avg = silhouette_score(graph_adjacency_matrix, cluster_labels)

        if silhouette_avg > best_silhouette_score:
            best_num_clusters = num_clusters
            best_cluster_labels = cluster_labels
            best_silhouette_score = silhouette_avg

    return best_num_clusters, best_cluster_labels

num_clusters_range = range(2, 5)

kendall_tau_matrix_without_GF[kendall_tau_matrix_without_GF < 0] = 0
best_num_clusters_withoutGF, cluster_labels_withoutGF = find_best_num_clusters(kendall_tau_matrix_without_GF, num_clusters_range)

print("Best number of clusters withoutGF:", best_num_clusters_withoutGF)
print("Cluster labels withoutGF:", cluster_labels_withoutGF)
print(len(cluster_labels_withoutGF))

In [ ]:
def get_top_k_features_per_cluster(cluster_labels, normalized_features_weight_vector, k_percentage):
    """
    Gets the indices of the top k% of features from each cluster based on their weights and cluster size,
    but only selects from clusters where the average weight of features is higher than the global average.

    Args:
        cluster_labels: A NumPy array of cluster labels for each feature.
        normalized_features_weight_vector: A NumPy array of normalized feature weights.
        k_percentage: The percentage of top features to select from each cluster.

    Returns:
        A dictionary where keys are cluster labels and values are lists of indices of the top k% features.
    """

    num_clusters = len(set(cluster_labels))
    top_k_features_per_cluster = {}
    global_avg_weight = np.mean(normalized_features_weight_vector)

    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_weights = normalized_features_weight_vector[cluster_indices]

        # Calculate the average weight of features in the cluster
        cluster_avg_weight = np.mean(cluster_weights)

        # Only select features from clusters with average weight above global average
        if cluster_avg_weight > global_avg_weight:
            # Calculate the number of features to select
            num_features_to_select = int(len(cluster_indices) * k_percentage / 100)

            # Get indices of top features within the cluster
            top_k_indices = np.argsort(cluster_weights)[::-1][:num_features_to_select]

            # Map indices back to original feature indices and adjust for 1-based indexing
            top_k_features_per_cluster[cluster_id] = cluster_indices[top_k_indices] + 1

    return top_k_features_per_cluster
#----------------------

def get_top_k_features_per_cluster_MAIN(cluster_labels, normalized_features_weight_vector, k_percentage):
    """
    Gets the indices of the top k% of features from each cluster based on their weights and cluster size.
    Feature indices start from 1.

    Args:
        cluster_labels: A NumPy array of cluster labels for each feature.
        normalized_features_weight_vector: A NumPy array of normalized feature weights.
        k_percentage: The percentage of top features to select from each cluster.

    Returns:
        A dictionary where keys are cluster labels and values are lists of indices of the top k% features.
    """

    num_clusters = len(set(cluster_labels))
    top_k_features_per_cluster = {}

    for cluster_id in range(num_clusters):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        cluster_weights = normalized_features_weight_vector[cluster_indices]

        # Calculate the number of features to select based on k_percentage and cluster size
        num_features_to_select = int(len(cluster_indices) * k_percentage / 100)

        # Get indices of top features within the cluster
        top_k_indices = np.argsort(cluster_weights)[::-1][:num_features_to_select]

        # Map indices back to original feature indices and adjust for 1-based indexing
        top_k_features_per_cluster[cluster_id] = cluster_indices[top_k_indices] + 1

    return top_k_features_per_cluster



def get_top_k_features(normalized_features_weight_vector, k_percentage):
    """
    Get the indices of the top k% features based on their weights.
    Treats all features as belonging to a single cluster.

    Parameters:
    normalized_features_weight_vector (numpy array): Array of normalized feature weights
    k_percentage (float): Percentage of top features to select (0-100)

    Returns:
    dict: Dictionary where key is 0 (single cluster) and value is list of indices
          of the top k% features (indices start from 1)
    """

    # Validate inputs
    if not isinstance(normalized_features_weight_vector, np.ndarray):
        raise ValueError("normalized_features_weight_vector must be a NumPy array")

    if k_percentage < 0 or k_percentage > 100:
        raise ValueError("k_percentage must be between 0 and 100")

    # Calculate how many top features to select (at least 1)
    n_features = len(normalized_features_weight_vector)
    k = max(1, int(n_features * k_percentage / 100))
    print("k_percentage:", k_percentage ,",  #selected features:",k)
    # Get indices of top k features (sorted by weight descending)
    top_k_indices = np.argsort(normalized_features_weight_vector)[::-1][:k]

    # Convert to feature indices (starting from 1)
    top_k_indices_1_based = (top_k_indices + 1).tolist()

    # Return as dictionary with single cluster (key = 0)
    return {0: top_k_indices_1_based}



k_percentage = 30  # Select top N% of features from each cluster-30

# Initialize an empty list to store the results
top_k_features_list = []

# Iterate over the last 10 columns of the matrix
for col in range(kendall_tau_matrix_without_rel.shape[1] - 22, kendall_tau_matrix_without_rel.shape[1]):
    # Extract the first 46 elements of the column
    normalized_features_weight_vector = kendall_tau_matrix_without_rel[:29, col]

    # Call the function to get top k features
    top_k_features = get_top_k_features_per_cluster_MAIN(cluster_labels_withoutGF, normalized_features_weight_vector, k_percentage)

    # Flatten the result (in case it's a nested list or array) and append to the list
    top_k_features_flat = np.array(top_k_features).flatten()
    top_k_features_list.append(top_k_features_flat)

# Convert the list to a numpy array
top_k_features_array = np.array(top_k_features_list)

print(top_k_features_array)

In [ ]:

df_test['doc_pagerank'] = 0
df_test['query_pagerank'] = 0
df_test['doc_eigenvector'] = 0
df_test['query_eigenvector'] = 0
df_test['doc_betweenness'] = 0
df_test['query_betweenness'] = 0
df_test['doc_closeness'] = 0
df_test['query_closeness'] = 0
df_test['doc_degree'] = 0
df_test['query_degree'] = 0
df_test['doc_clustering'] = 0
df_test['query_clustering'] = 0
df_test['doc_avg_neighbor_deg'] = 0
df_test['query_avg_neighbor_deg'] = 0
df_test['doc_common_neighbors_ratio'] = 0
df_test['query_common_neighbors_ratio'] = 0
df_test['doc_weighted_common_neighbors'] = 0
df_test['query_weighted_common_neighbors'] = 0
df_test['doc_projection_degree'] = 0
df_test['doc_projection_clustering'] = 0
df_test['query_projection_degree'] = 0
df_test['query_projection_clustering'] = 0

##Predition of All Graph features with DL

In [ ]:


# Set all seeds for maximum reproducibility
SEED = 123
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Clear any previous sessions
tf.keras.backend.clear_session()

def convert_to_feature_names(input_dict):
    feature_names = []
    for feature_id, indices in input_dict.items():
        for index in indices:
            feature_names.append(index)
    return feature_names

def extract_columns_by_feature_names(df_input, features_name, top_k_features_ids):
    selected_feature_names = [features_name[i - 1] for i in top_k_features_ids]
    extracted_df = df_input[selected_feature_names]
    return extracted_df


def build_wide_deep_model(input_dim):
    """
    Wide & Deep architecture - combines memorization and generalization
    """
    inputs = keras.Input(shape=(input_dim,))

    # Wide path (direct connections)
    wide_branch = layers.BatchNormalization()(inputs)
    wide_branch = layers.Dense(8, activation='relu')(wide_branch)

    # Deep path
    deep_branch = layers.BatchNormalization()(inputs)
    deep_branch = layers.Dense(128, activation='relu')(deep_branch)
    deep_branch = layers.BatchNormalization()(deep_branch)
    deep_branch = layers.Dropout(0.3)(deep_branch)

    deep_branch = layers.Dense(64, activation='relu')(deep_branch)
    deep_branch = layers.BatchNormalization()(deep_branch)
    deep_branch = layers.Dropout(0.3)(deep_branch)

    deep_branch = layers.Dense(32, activation='relu')(deep_branch)
    deep_branch = layers.BatchNormalization()(deep_branch)

    deep_branch = layers.Dense(64, activation='relu')(deep_branch)
    deep_branch = layers.BatchNormalization()(deep_branch)

    # Concatenate wide and deep paths
    concatenated = layers.Concatenate()([wide_branch, deep_branch])

    # Final layers
    x = layers.Dense(32, activation='relu')(concatenated)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(1, activation='linear')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )

    return model

#@@@@@@@@@@@@@@@@@@@@@@
def build_improved_model(input_dim):
    """
    Simplified model with strong regularization to prevent overfitting
    """
    # Initializers with seed
    kernel_init = tf.keras.initializers.GlorotUniform(seed=SEED)
    bias_init = tf.keras.initializers.Zeros()

    inputs = keras.Input(shape=(input_dim,))

    # Input normalization
    x = layers.BatchNormalization()(inputs)

    # First block with strong regularization
    x = layers.Dense(64, activation='relu',
                    kernel_initializer=kernel_init,
                    kernel_regularizer=l2(0.01),
                    bias_initializer=bias_init)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)  # High dropout

    # Second block
    x = layers.Dense(32, activation='relu',
                    kernel_initializer=kernel_init,
                    kernel_regularizer=l2(0.01),
                    bias_initializer=bias_init)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    # Third block
    x = layers.Dense(16, activation='relu',
                    kernel_initializer=kernel_init,
                    kernel_regularizer=l2(0.005),
                    bias_initializer=bias_init)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Output layer
    outputs = layers.Dense(1, activation='linear')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    # Optimizer with lower learning rate
    optimizer = keras.optimizers.Adam(
        learning_rate=0.0005,  # Lower learning rate
        beta_1=0.9,
        beta_2=0.999,
        clipnorm=1.0
    )

    model.compile(
        optimizer=optimizer,
        loss='mse',  # Simple MSE for stability
        metrics=["mae"]
    )

    return model

def create_callbacks():
    """Create training callbacks with early stopping"""
    return [
        EarlyStopping(
            monitor='val_loss',
            patience=15,  # More patience
            restore_best_weights=True,
            mode='min',
            verbose=1,
            min_delta=0.001  # Minimum improvement
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=8,
            min_lr=1e-6,
            verbose=1
        )
    ]

def preprocess_data(X_train,X_test):
    """Data preprocessing"""
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, scaler

def predict_centrality_features(df_train, df_test, top_k_features_array):
    """
    Enhanced prediction function with better training and validation
    """
    target_cols = df_train.columns[-22:]

    # Initialize results
    models = {}
    scalers = {}

    for j in range(0,22):
        for  i,dictionary in enumerate(top_k_features_array[j]):
            if j >= len(target_cols):
                break

            top_k_features_ids = convert_to_feature_names(dictionary)
            features_name_all = df_train.columns[:]
            features_name = features_name_all.drop(['qid', 'docid', 'relevance'])

            # Extract features
            X_train = extract_columns_by_feature_names(df_train, features_name, top_k_features_ids)
            y_train = df_train[target_cols[j]]
            X_test = extract_columns_by_feature_names(df_test, features_name, top_k_features_ids)

            # Preprocess data
            X_train_processed,  X_test_processed, scaler = preprocess_data(
                X_train, X_test
            )
            scalers[j] = scaler

            # Build and train model
            model = build_improved_model(X_train_processed.shape[1])

            print(f"\nTraining model for target {target_cols[j]} ({j+1}/{len(target_cols)})")
            print(f"Input shape: {X_train_processed.shape}")

            # Train with validation
            history = model.fit(
                X_train_processed, y_train,
                epochs=50,
                batch_size=32,
                validation_split=0.11,
                verbose=0,
                callbacks=create_callbacks(),
                shuffle=False
            )

            # Store model
            models[j] = model

            # Predict on test set
            test_predictions = model.predict(X_test_processed, verbose=0).flatten()
            df_test[target_cols[j]] = test_predictions

            # Print training summary
            best_epoch = np.argmin(history.history['val_loss'])
            print(f"Completed {target_cols[i]} - Best val loss: {history.history['val_loss'][best_epoch]:.4f} (epoch {best_epoch+1})")
            print(f"Training loss: {history.history['loss'][best_epoch]:.4f}, Val loss: {history.history['val_loss'][best_epoch]:.4f}")



    return df_test, models, scalers


print("Starting centrality feature prediction...")

# Make predictions with improved model
df_test_with_centralities, trained_models, feature_scalers = predict_centrality_features(
    df_train_with_centralities,
    df_test,top_k_features_array)
# Copy evaluation script
source_path = '/content/drive/MyDrive/Eval-Score-4.0-WCL2R.pl'  # Replace with actual path
destination_path = '/content/Eval-Score-4.0-WCL2R.pl'
# Copy the file
!cp {source_path} {destination_path}

  #------------------------------------------------------
# Open the output file in write mode
with open(f"/content/judgements.txt", "w") as outfile:
    # Iterate over each row in the DataFrame
    for index, row in df_test_with_centralities.iterrows():
        # Extract the required values
        relevance_value = int(row['relevance'])
        qid_value = int(row['qid'])
        docid_value = int(row['docid'])

        # Format the line as per the requirement
        line = f"{relevance_value} qid:{qid_value} #docid = {docid_value}\n"

        # Write the line to the file
        outfile.write(line)
# Prepare features for evaluation
n_features = 51
p_at_1_test = pd.Series(np.zeros(n_features))
df_test_withoutqid_docid = df_test_with_centralities[features_df].astype(float)

for i in range(1, n_features + 1):
    if i >= len(df_test_withoutqid_docid.columns):
        break

    predicted_labels = df_test_withoutqid_docid.iloc[:, i]

    with open('predictions-F1.txt', 'w') as f:
        for f1 in predicted_labels:
            f.write(f"  {f1:.7e}  \n")

    # Run evaluation
    !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions-F1.txt output.txt 0
    # Read results

    df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
    p_at_1_test[i-1] = float(df.iloc[1, 1])
print("\n" + "="*50)
print("EVALUATION RESULTS SUMMARY")
print("="*50)
print(f"P@1 results: {p_at_1_test[35]}")

df_test_with_centralities.to_csv(
    'test_with_centralities_DL22.csv',
    index=False,
    encoding='utf-8'
    ,float_format='%.8f'
)


##Apply weight to features

In [ ]:
sc2=StandardScaler()
df_train_with_centralities[graph_feature_cols] = sc2.fit_transform(df_train_with_centralities[graph_feature_cols])
sc=StandardScaler()
df_test_with_centralities[graph_feature_cols] = sc.fit_transform(df_test_with_centralities[graph_feature_cols])
sc3=StandardScaler()

# Prepare features for evaluation
n_features = 51
p_at_1_test = pd.Series(np.zeros(n_features))
df_test_withoutqid_docid = df_test_with_centralities[features_df].astype(float)

# Open the output file in write mode
with open(f"/content/judgements.txt", "w") as outfile:
    # Iterate over each row in the DataFrame
    for index, row in df_test_with_centralities.iterrows():
        # Extract the required values
        relevance_value = int(row['relevance'])
        qid_value = int(row['qid'])
        docid_value = int(row['docid'])

        # Format the line as per the requirement
        line = f"{relevance_value} qid:{qid_value} #docid = {docid_value}\n"

        # Write the line to the file
        outfile.write(line)

for i in range(1, n_features + 1):
    if i >= len(df_test_withoutqid_docid.columns):
        break

    predicted_labels = df_test_withoutqid_docid.iloc[:, i]

    with open('predictions-F1.txt', 'w') as f:
        for f1 in predicted_labels:
            f.write(f"  {f1:.7e}  \n")

    # Run evaluation
    !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions-F1.txt output.txt 0
    # Read results

    df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
    p_at_1_test[i-1] = float(df.iloc[1, 1])
print("\n" + "="*50)
print("EVALUATION RESULTS SUMMARY")
print("="*50)
print(f"P@1 results: {p_at_1_test}")

## Prepare Train and Test Datasets

In [ ]:
X_train = df_train_with_centralities.drop(columns=['docid','qid','relevance'])
y_train = df_train_with_centralities[['relevance']].astype(int)
X_test = df_test_with_centralities.drop(columns=['docid','qid','relevance'])
y_test = df_test_with_centralities[['relevance']].astype(int)

## Select Top K% of features based on P@!(Train)

In [ ]:
k =30
# Calculate number of elements to select
n = int(len(p_at_1_train) * k / 100)
print(n)
# Get indexes of top k% values(add 1)
top_k_percent_indexes = [x + 1 for x in p_at_1_train.nlargest(n).index.tolist()]

print(top_k_percent_indexes)

In [ ]:
def convert_to_feature_names(input_dict):

  feature_names = []
  for featrue_id, indices in input_dict.items():
    for index in indices:
      #feature_names.append(f"{index}")
      feature_names.append(index)
  return feature_names


def extract_columns_by_feature_names(df_input, features_name, top_k_features_ids):
    """
    Extracts columns from a DataFrame based on feature indices.

    Args:
        df_train: The input DataFrame.
        features_name: A list of feature names.
        top_k_features_ids: A list of feature indices.

    Returns:
        A new DataFrame containing only the extracted columns.
    """

    # Extract feature names corresponding to the top indices
    selected_feature_names = [features_name[i - 1] for i in top_k_features_ids]

    # Extract the corresponding columns from the DataFrame
    extracted_df = df_input[selected_feature_names]
    print(selected_feature_names)
    return extracted_df


features_name_all = df_train_with_centralities.columns[:]
features_name = features_name_all.drop(['qid'])
features_name = features_name.drop(['docid'])
features_name = features_name.drop(['relevance'])
print(features_name)

X_train = extract_columns_by_feature_names(df_train_with_centralities, features_name, top_k_percent_indexes)
y_train = df_train.iloc[:, 0].astype(int)


X_test = extract_columns_by_feature_names(df_test_with_centralities, features_name, top_k_percent_indexes).astype(int)
y_test = df_test.iloc[:, 0].astype(int)

##Classic Models for prediction

In [ ]:
path="/content"
filename = "output.txt"
output_full_path = os.path.join(path, filename)
with open(output_full_path, 'w'):
    pass

#Create judgements file
with open(f"/content/judgements.txt", "w") as outfile:
    for index, row in df_test_with_centralities.iterrows():
        # Extract the required values
        relevance_value = int(row['relevance'])
        qid_value = int(row['qid'])
        docid_value = int(row['docid'])

        # Format the line as per the requirement
        line = f"{relevance_value} qid:{qid_value} #docid = {docid_value}\n"

        # Write the line to the file
        outfile.write(line)

"""
xbgReg_model = XGBRegressor(n_estimators=25, learning_rate=0.12, max_depth=3, random_state=42)
xbgReg_model.fit(X_train, y_train)
predicted_labels = xbgReg_model.predict(X_test)
"""


"""
# Define base models
base_models = [
  ('lgbm', LGBMRegressor(n_estimators=30, max_depth=3, min_child_samples=30, verbosity=-1, random_state=42)),
  ('gbr', GradientBoostingRegressor(n_estimators=1, min_samples_split=5, random_state=42))
]

# Define the meta-model (final estimator)
meta_model = Ridge(random_state=42)

# Create the StackingRegressor
model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model
)
model.fit(X_train, y_train)
predicted_labels = model.predict(X_test)
"""

'''
lgbReg_model = LGBMRegressor(n_estimators=1, learning_rate=0.11, max_depth=3, min_child_samples=10, random_state=42, verbosity=-1)
lgbReg_model.fit(X_train, y_train)
'''
'''
#------------------------------------LGBMRegressor-----------------------------------
# Define the parameter grid for LGBM Regressor
param_grid = {
    'n_estimators': list(range(1, 100, 1)),  # 1 to 200, step 5
    'max_depth': list(range(1,5, 1)),  # 1 to 30, step 1
    'min_child_samples': list(range(1, 11, 1)),  # 10 to 15, step 1 (equivalent to min_samples_split)
    'num_leaves': [31],  # Default value, you can adjust this if needed
    'learning_rate': [0.1],  # Default learning rate
    'subsample': [1.0],  # Default subsample
    'colsample_bytree': [1.0],  # Default feature fraction
    #'n_estimators': list(range(49, 50, 1)),  # 1 to 200, step 5
    #'max_depth': list(range(2, 3, 1)),  # 1 to 30, step 1
    #'min_child_samples': list(range(1, 2, 1)),  # 10 to 15, step 1 (equivalent to min_samples_split)

}

# Generate all parameter combinations
all_params = list(ParameterGrid(param_grid))
print(f"Total parameter combinations to test: {len(all_params)}")

best_p_at_1 = 0
best_params = None
best_model = None
results = []

# Grid search
for i, params in enumerate(all_params):
    print(f"Testing combination {i+1}/{len(all_params)}: n_estimators={params['n_estimators']}, max_depth={params['max_depth']}, min_child_samples={params['min_child_samples']}")

    try:
        # Train LGBM model with current parameters
        lgbm_model = LGBMRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_child_samples=params['min_child_samples'],
            num_leaves=params['num_leaves'],
            learning_rate=params['learning_rate'],
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            n_jobs=-1,  # Use all available cores for faster computation
            random_state=42,  # For reproducibility
            verbose=-1  # Suppress LightGBM output
        )

        lgbm_model.fit(X_train, y_train)
        predicted_labels = lgbm_model.predict(X_test)

        # Create predictions file
        with open(f'predictions.txt', 'w') as f:
            for f1 in predicted_labels:
                f.write(f"  {f1:.7e}  \n")

        # Run evaluation
        !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions.txt output.txt 0

        # Read results
        df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
        current_p_at_1 = float(df.iloc[1, 1])

        # Store results
        results.append({
            'n_estimators': params['n_estimators'],
            'max_depth': params['max_depth'],
            'min_child_samples': params['min_child_samples'],
            'num_leaves': params['num_leaves'],
            'learning_rate': params['learning_rate'],
            'subsample': params['subsample'],
            'colsample_bytree': params['colsample_bytree'],
            'p_at_1': current_p_at_1
        })

        print(f"P@1: {current_p_at_1}")

        # Update best parameters if current is better
        if current_p_at_1 > best_p_at_1:
            best_p_at_1 = current_p_at_1
            best_params = params.copy()
            best_model = lgbm_model
            print(f"⭐ New best! P@1: {best_p_at_1:.6f} with n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_child_samples={best_params['min_child_samples']}")

    except Exception as e:
        print(f"Error with parameters {params}: {e}")
        continue

# Final results
print("\n" + "="*80)
print("LGBM REGRESSOR GRID SEARCH COMPLETE")
print("="*80)
print(f"Best P@1: {best_p_at_1:.6f}")
print(f"Best parameters: n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_child_samples={best_params['min_child_samples']}, num_leaves={best_params['num_leaves']}, learning_rate={best_params['learning_rate']}")
print("="*80)

# Save results to CSV for analysis
results_df = pd.DataFrame(results)
results_df.to_csv('lgbm_regressor_grid_search_results.csv', index=False)
print("Results saved to 'lgbm_regressor_grid_search_results.csv'")

# Save the best model
joblib.dump(best_model, 'best_lgbm_regressor_model.pkl')
print("Best model saved as 'best_lgbm_regressor_model.pkl'")

# Display top 10 results
print("\nTop 10 parameter combinations:")
top_results = results_df.nlargest(10, 'p_at_1')
for i, (_, row) in enumerate(top_results.iterrows(), 1):
    print(f"{i}. P@1: {row['p_at_1']:.6f} - n_estimators={row['n_estimators']}, max_depth={row['max_depth']}, min_child_samples={row['min_child_samples']}")
'''

'''
# Define the parameter grid for XGBRegressor
param_grid = {
    'n_estimators': list(range(60, 81, 5)),  # 10 to 500, step 50
    'learning_rate': [round(x, 2) for x in np.arange(0.06, 0.08, 0.01)],
    'max_depth': list(range(1, 3)),  # 1 to 5, step 1
    'min_child_weight': [20, 30, 40]  # min_child_samples equivalent in XGBoost
}

# Generate all parameter combinations
all_params = list(ParameterGrid(param_grid))
print(f"Total parameter combinations to test: {len(all_params)}")

best_p_at_1 = 0
best_params = None
best_model = None
results = []

# Grid search
for i, params in enumerate(all_params):
    print(f"Testing combination {i+1}/{len(all_params)}: {params}")

    try:
        # Train XGBRegressor model with current parameters
        xgb_model = XGBRegressor(
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            max_depth=params['max_depth'],
            min_child_weight=params['min_child_weight'],
            random_state=42,
            verbosity=0,
            n_jobs=-1  # Use all available cores
        )

        xgb_model.fit(X_train, y_train)
        predicted_labels = xgb_model.predict(X_test)

        # Create judgements file
        with open(f"/content/drive/MyDrive/LETOR4.0/MQ2007/test.txt", "r") as infile, open(f"/content/judgements.txt", "w") as outfile:
            for line in infile:
                parts = line.split()
                label = parts[0]
                qid_part = parts[1]
                docid_part = parts[-7]
                inc_part = parts[-4]
                prob_part = parts[-1]
                outfile.write(f"{parts[0]} {qid_part} #docid = {docid_part} inc = {inc_part} prob = {prob_part}\n")

        # Create predictions file
        with open(f'predictions.txt', 'w') as f:
            for f1 in predicted_labels:
                f.write(f"  {f1:.7e}  \n")

        # Run evaluation
        !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions.txt output.txt 0

        # Read results
        df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
        current_p_at_1 = float(df.iloc[1, 1])

        # Store results
        results.append({
            'n_estimators': params['n_estimators'],
            'learning_rate': params['learning_rate'],
            'max_depth': params['max_depth'],
            'min_child_weight': params['min_child_weight'],
            'p_at_1': current_p_at_1
        })

        print(f"P@1: {current_p_at_1:.6f}")

        # Update best parameters if current is better
        if current_p_at_1 > best_p_at_1:
            best_p_at_1 = current_p_at_1
            best_params = params.copy()
            best_model = xgb_model
            print(f"⭐ New best! P@1: {best_p_at_1:.6f} with params: {best_params}")

    except Exception as e:
        print(f"Error with parameters {params}: {e}")
        continue

# Final results
print("\n" + "="*70)
print("XGBREGRESSOR GRID SEARCH COMPLETE")
print("="*70)
print(f"Best P@1: {best_p_at_1:.6f}")
print(f"Best parameters: {best_params}")
print("="*70)

# Save results to CSV for analysis
results_df = pd.DataFrame(results)
results_df.to_csv('xgboost_regressor_grid_search_results.csv', index=False)
print("Results saved to 'xgboost_regressor_grid_search_results.csv'")

# Save the best model
joblib.dump(best_model, 'best_xgboost_regressor_model.pkl')
print("Best model saved as 'best_xgboost_regressor_model.pkl'")

# Display top 10 results
print("\nTop 10 parameter combinations:")
top_results = results_df.nlargest(10, 'p_at_1')
for i, (_, row) in enumerate(top_results.iterrows(), 1):
    print(f"{i}. P@1: {row['p_at_1']:.6f} - n_estimators={row['n_estimators']}, lr={row['learning_rate']}, max_depth={row['max_depth']}, min_child_weight={row['min_child_weight']}")

# Additional analysis - group by parameters to see trends
print("\nAverage P@1 by parameter:")
print("n_estimators:")
print(results_df.groupby('n_estimators')['p_at_1'].mean().sort_values(ascending=False).head())

print("\nlearning_rate:")
print(results_df.groupby('learning_rate')['p_at_1'].mean().sort_values(ascending=False).head())

print("\nmax_depth:")
print(results_df.groupby('max_depth')['p_at_1'].mean().sort_values(ascending=False).head())

print("\nmin_child_weight:")
print(results_df.groupby('min_child_weight')['p_at_1'].mean().sort_values(ascending=False).head())

'''

#-----------------------------------KNN with Parameters-------------------------------------------------------------
# Define the parameter grid for KNN Regressor
param_grid = {
    #'n_neighbors': list(range(1, 301, 5)),  # 50 to 500, step 50
    #'p': list(range(1, 3))  # 1 to 3, step 1 (1=Manhattan, 2=Euclidean, 3=Minkowski)
    'n_neighbors': list(range(266,267, 1)),  #385-400(Not DL)
    'p': list(range(2, 3))  # 1 to 3, step 1 (1=Manhattan, 2=Euclidean, 3=Minkowski)p=1(Not DL)

}

# Generate all parameter combinations
all_params = list(ParameterGrid(param_grid))
print(f"Total parameter combinations to test: {len(all_params)}")

best_p_at_1 = 0
best_params = None
best_model = None
results = []

# Grid search
for i, params in enumerate(all_params):
    print(f"Testing combination {i+1}/{len(all_params)}: n_neighbors={params['n_neighbors']}, p={params['p']}")

    try:
        # Train KNN model with current parameters

        knn_model = KNeighborsRegressor(
            n_neighbors=params['n_neighbors'],
            p=params['p'],
            weights='distance',
            n_jobs=-1  # Use all available cores for faster computation
        )
        knn_model.fit(X_train, y_train)
        predicted_labels = knn_model.predict(X_test)

        # Create predictions file
        with open(f'predictions.txt', 'w') as f:
            for f1 in predicted_labels:
                f.write(f"  {f1:.7e}  \n")

        # Run evaluation
        !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions.txt output.txt 0

        # Read results
        df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
        current_p_at_1 = float(df.iloc[1, 1])
        print(df)

        # Store results
        results.append({
            'n_neighbors': params['n_neighbors'],
            'p': params['p'],
            'p_at_1': current_p_at_1
        })

        print(f"P@1: {current_p_at_1}")

        # Update best parameters if current is better
        if current_p_at_1 > best_p_at_1:
            best_p_at_1 = current_p_at_1
            best_params = params.copy()
            best_model = knn_model
            #print(f"⭐ New best! P@1: {best_p_at_1:.6f} with n_neighbors={best_params['n_neighbors']}, p={best_params['p']}")

    except Exception as e:
        print(f"Error with parameters {params}: {e}")
        continue

# Final results
print("\n" + "="*60)
print("KNN REGRESSOR GRID SEARCH COMPLETE")
print("="*60)
print(f"Best P@1: {best_p_at_1:.6f}")
print("="*60)

# Save results to CSV for analysis
results_df = pd.DataFrame(results)
results_df.to_csv('knn_regressor_grid_search_results.csv', index=False)
print("Results saved to 'knn_regressor_grid_search_results.csv'")

# Save the best model
joblib.dump(best_model, 'best_knn_regressor_model.pkl')
print("Best model saved as 'best_knn_regressor_model.pkl'")

# Display top 10 results
print("\nTop 10 parameter combinations:")
top_results = results_df.nlargest(10, 'p_at_1')
for i, (_, row) in enumerate(top_results.iterrows(), 1):
    print(f"{i}. P@1: {row['p_at_1']:.6f} - n_neighbors={row['n_neighbors']}, p={row['p']}")



'''
# Define the parameter grid for XGB Regressor
param_grid = {
    #'n_estimators': list(range(1, 101, 1)),  # 1 to 200, step 5
    #'max_depth': list(range(1, 5, 1)),  # 1 to 30, step 1
    #'min_child_weight': list(range(1, 11, 1)),  # 1 to 5, step 1 (equivalent to min_samples_split)
    'learning_rate': [0.1],  # Default learning rate
    'subsample': [1.0],  # Default subsample
    'colsample_bytree': [1.0],  # Default feature fraction
    'gamma': [0],  # Default minimum loss reduction
    'n_estimators': list(range(97, 98, 1)),  # 1 to 200, step 5
    'max_depth': list(range(2, 3, 1)),  # 1 to 30, step 1
    'min_child_weight': list(range(1, 2, 1)),  # 1 to 5, step 1 (equivalent to min_samples_split)

}

# Generate all parameter combinations
all_params = list(ParameterGrid(param_grid))
print(f"Total parameter combinations to test: {len(all_params)}")

best_p_at_1 = 0
best_params = None
best_model = None
results = []

# Grid search
for i, params in enumerate(all_params):
    print(f"Testing combination {i+1}/{len(all_params)}: n_estimators={params['n_estimators']}, max_depth={params['max_depth']}, min_child_weight={params['min_child_weight']}")

    try:
        # Train XGB model with current parameters
        xgb_model = XGBRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_child_weight=params['min_child_weight'],
            learning_rate=params['learning_rate'],
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            gamma=params['gamma'],
            n_jobs=-1,  # Use all available cores for faster computation
            random_state=42,  # For reproducibility
            verbosity=0  # Suppress XGBoost output
        )

        xgb_model.fit(X_train, y_train)
        predicted_labels = xgb_model.predict(X_test)

        # Create predictions file
        with open(f'predictions.txt', 'w') as f:
            for f1 in predicted_labels:
                f.write(f"  {f1:.7e}  \n")

        # Run evaluation
        !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions.txt output.txt 0

        # Read results
        df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
        current_p_at_1 = float(df.iloc[1, 1])

        # Store results
        results.append({
            'n_estimators': params['n_estimators'],
            'max_depth': params['max_depth'],
            'min_child_weight': params['min_child_weight'],
            'learning_rate': params['learning_rate'],
            'subsample': params['subsample'],
            'colsample_bytree': params['colsample_bytree'],
            'gamma': params['gamma'],
            'p_at_1': current_p_at_1
        })

        print(f"P@1: {current_p_at_1}")

        # Update best parameters if current is better
        if current_p_at_1 > best_p_at_1:
            best_p_at_1 = current_p_at_1
            best_params = params.copy()
            best_model = xgb_model
            print(f"⭐ New best! P@1: {best_p_at_1:.6f} with n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_child_weight={best_params['min_child_weight']}")

    except Exception as e:
        print(f"Error with parameters {params}: {e}")
        continue

# Final results
print("\n" + "="*80)
print("XGB REGRESSOR GRID SEARCH COMPLETE")
print("="*80)
print(f"Best P@1: {best_p_at_1:.6f}")
print(f"Best parameters: n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_child_weight={best_params['min_child_weight']}, learning_rate={best_params['learning_rate']}, subsample={best_params['subsample']}, colsample_bytree={best_params['colsample_bytree']}, gamma={best_params['gamma']}")
print("="*80)

# Save results to CSV for analysis
results_df = pd.DataFrame(results)
results_df.to_csv('xgb_regressor_grid_search_results.csv', index=False)
print("Results saved to 'xgb_regressor_grid_search_results.csv'")

# Save the best model
joblib.dump(best_model, 'best_xgb_regressor_model.pkl')
print("Best model saved as 'best_xgb_regressor_model.pkl'")

# Display top 10 results
print("\nTop 10 parameter combinations:")
top_results = results_df.nlargest(10, 'p_at_1')
for i, (_, row) in enumerate(top_results.iterrows(), 1):
    print(f"{i}. P@1: {row['p_at_1']:.6f} - n_estimators={row['n_estimators']}, max_depth={row['max_depth']}, min_child_weight={row['min_child_weight']}")

'''
'''
# Define the parameter grid for RandomForest Regressor
param_grid = {
    #'n_estimators': list(range(1, 101, 1)),  # 1 to 200, step 5
    #'max_depth': list(range(1, 11, 1)),  # 1 to 30, step 1
    #'min_samples_split': list(range(1, 31, 1)),  # 10 to 15, step 1
    'max_features': ['log2'],
    'min_samples_leaf': [1],
    'n_estimators': [50],  # 1 to 200, step 5
    'max_depth': [2],  # 1 to 30, step 1 18
    'min_samples_split': [10],  # 10 to 15, step 1 10

}

# Generate all parameter combinations
all_params = list(ParameterGrid(param_grid))
print(f"Total parameter combinations to test: {len(all_params)}")

best_p_at_1 = 0
best_params = None
best_model = None
results = []

# Grid search
for i, params in enumerate(all_params):
    print(f"Testing combination {i+1}/{len(all_params)}: n_estimators={params['n_estimators']}, max_depth={params['max_depth']}, min_samples_split={params['min_samples_split']}")

    try:
        # Train RandomForest model with current parameters
        rf_model = RandomForestRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_samples_split=params['min_samples_split'],
            max_features=params['max_features'],
            min_samples_leaf=params['min_samples_leaf'],
            n_jobs=-1,  # Use all available cores for faster computation
            random_state=42  # For reproducibility
        )

        rf_model.fit(X_train, y_train)
        predicted_labels = rf_model.predict(X_test)

        # Create predictions file
        with open(f'predictions.txt', 'w') as f:
            for f1 in predicted_labels:
                f.write(f"  {f1:.7e}  \n")

        # Run evaluation
        !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions.txt output.txt 0

        # Read results
        df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
        current_p_at_1 = float(df.iloc[1, 1])

        # Store results
        results.append({
            'n_estimators': params['n_estimators'],
            'max_depth': params['max_depth'],
            'min_samples_split': params['min_samples_split'],
            'max_features': params['max_features'],
            'min_samples_leaf': params['min_samples_leaf'],
            'p_at_1': current_p_at_1
        })

        print(f"P@1: {current_p_at_1}")

        # Update best parameters if current is better
        if current_p_at_1 > best_p_at_1:
            best_p_at_1 = current_p_at_1
            best_params = params.copy()
            best_model = rf_model
            print(f"⭐ New best! P@1: {best_p_at_1:.6f} with n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_samples_split={best_params['min_samples_split']}")

    except Exception as e:
        print(f"Error with parameters {params}: {e}")
        continue

# Final results
print("\n" + "="*80)
print("RANDOM FOREST REGRESSOR GRID SEARCH COMPLETE")
print("="*80)
print(f"Best P@1: {best_p_at_1:.6f}")
print(f"Best parameters: n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_samples_split={best_params['min_samples_split']}, max_features={best_params['max_features']}, min_samples_leaf={best_params['min_samples_leaf']}")
print("="*80)

# Save results to CSV for analysis
results_df = pd.DataFrame(results)
results_df.to_csv('random_forest_regressor_grid_search_results.csv', index=False)
print("Results saved to 'random_forest_regressor_grid_search_results.csv'")

# Save the best model
joblib.dump(best_model, 'best_random_forest_regressor_model.pkl')
print("Best model saved as 'best_random_forest_regressor_model.pkl'")

# Display top 10 results
print("\nTop 10 parameter combinations:")
top_results = results_df.nlargest(10, 'p_at_1')
for i, (_, row) in enumerate(top_results.iterrows(), 1):
    print(f"{i}. P@1: {row['p_at_1']:.6f} - n_estimators={row['n_estimators']}, max_depth={row['max_depth']}, min_samples_split={row['min_samples_split']}")
'''

'''
# Define the parameter grid for GradientBoosting Regressor
param_grid = {
    'n_estimators': list(range(1, 201, 5)),  # 1 to 200, step 5
    'max_depth': [None],#list(range(1, 4, 1)),  # 1 to 30, step 1
    #'min_samples_split': list(range(10, 12, 1)),  # 10 to 15, step 1
    #'learning_rate': [0.1, 0.05, 0.01],  # Common learning rates
    'subsample': [1.0],  # Subsample ratio
    'max_features': ['log2'],  # Feature sampling
    #'n_estimators': list(range(31, 32, 1)),
    #'max_depth': list(range(3, 4, 1)),
    'min_samples_split': list(range(10, 11, 1)),
    'learning_rate': [0.1],

}

# Generate all parameter combinations
all_params = list(ParameterGrid(param_grid))
print(f"Total parameter combinations to test: {len(all_params)}")

best_p_at_1 = 0
best_params = None
best_model = None
results = []

# Grid search
for i, params in enumerate(all_params):
    print(f"Testing combination {i+1}/{len(all_params)}: n_estimators={params['n_estimators']}, max_depth={params['max_depth']}, min_samples_split={params['min_samples_split']},learning_rate={params['learning_rate'] }")

    try:
        # Train GradientBoosting model with current parameters
        gb_model = GradientBoostingRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_samples_split=params['min_samples_split'],
            learning_rate=params['learning_rate'],
            subsample=params['subsample'],
            max_features=params['max_features'],
            random_state=42,  # For reproducibility
            verbose=0  # Suppress output
        )

        gb_model.fit(X_train, y_train)
        predicted_labels = gb_model.predict(X_test)


        # Create predictions file
        with open(f'predictions.txt', 'w') as f:
            for f1 in predicted_labels:
                f.write(f"  {f1:.7e}  \n")

        # Run evaluation
        !perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions.txt output.txt 0

        # Read results
        df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
        current_p_at_1 = float(df.iloc[1, 1])

        # Store results
        results.append({
            'n_estimators': params['n_estimators'],
            'max_depth': params['max_depth'],
            'min_samples_split': params['min_samples_split'],
            'learning_rate': params['learning_rate'],
            'subsample': params['subsample'],
            'max_features': params['max_features'],
            'p_at_1': current_p_at_1
        })

        print(f"P@1: {current_p_at_1}")

        # Update best parameters if current is better
        if current_p_at_1 > best_p_at_1:
            best_p_at_1 = current_p_at_1
            best_params = params.copy()
            best_model = gb_model
            print(f"⭐ New best! P@1: {best_p_at_1:.6f} with n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_samples_split={best_params['min_samples_split']}")

    except Exception as e:
        print(f"Error with parameters {params}: {e}")
        continue

# Final results
print("\n" + "="*80)
print("GRADIENT BOOSTING REGRESSOR GRID SEARCH COMPLETE")
print("="*80)
print(f"Best P@1: {best_p_at_1:.6f}")
print(f"Best parameters: n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, min_samples_split={best_params['min_samples_split']}, learning_rate={best_params['learning_rate']}, subsample={best_params['subsample']}, max_features={best_params['max_features']}")
print("="*80)

# Save results to CSV for analysis
results_df = pd.DataFrame(results)
results_df.to_csv('gradient_boosting_regressor_grid_search_results.csv', index=False)
print("Results saved to 'gradient_boosting_regressor_grid_search_results.csv'")

# Save the best model
joblib.dump(best_model, 'best_gradient_boosting_regressor_model.pkl')
print("Best model saved as 'best_gradient_boosting_regressor_model.pkl'")

# Display top 10 results
print("\nTop 10 parameter combinations:")
top_results = results_df.nlargest(10, 'p_at_1')
for i, (_, row) in enumerate(top_results.iterrows(), 1):
    print(f"{i}. P@1: {row['p_at_1']:.6f} - n_estimators={row['n_estimators']}, max_depth={row['max_depth']}, min_samples_split={row['min_samples_split']}")
'''


'''
gbr_model = GradientBoostingRegressor(n_estimators=10, max_depth=3, min_samples_split=10,learning_rate=0.1, random_state=42)
gbr_model.fit(X_train, y_train)
predicted_labels = gbr_model.predict(X_test)
'''
'''
knn_model = KNeighborsRegressor(n_neighbors=140, p=2) # n_neighbors=220, p=2 k=30 :: 0.4586
knn_model.fit(X_train, y_train)
predicted_labels = knn_model.predict(X_test)
'''

'''
lgbReg_model = LGBMRegressor(n_estimators=9, learning_rate=0.1, max_depth=2, min_child_samples=10, random_state=42, verbosity=-1)
lgbReg_model.fit(X_train, y_train)
predicted_labels = lgbReg_model.predict(X_test)
'''


'''
# Create judgements file
with open(f"/content/judgements.txt", "w") as outfile:
    for index, row in df_test_with_centralities.iterrows():
        # Extract the required values
        relevance_value = int(row['relevance'])
        qid_value = int(row['qid'])
        docid_value = int(row['docid'])

        # Format the line as per the requirement
        line = f"{relevance_value} qid:{qid_value} #docid = {docid_value}\n"

        # Write the line to the file
        outfile.write(line)

with open(f'predictions.txt', 'w') as f:
    for f1 in predicted_labels:
        f.write(f"  {f1:.7e}  \n")
!perl Eval-Score-4.0-WCL2R.pl judgements.txt predictions.txt output.txt 0
df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
p_at_1 = float(df.iloc[1, 1])
print(p_at_1, "\n\n")
'''

##Data Fusion

In [ ]:

def copy_files_from_drive(drive_folder_path, colab_destination='/content/'):
    """
    Copies files from Google Drive to Colab runtime directory and returns their paths.

    Parameters:
    drive_folder_path (str): Path to folder in Google Drive (e.g., '/content/gdrive/MyDrive/LDLTR/WCL2R')
    colab_destination (str): Destination directory in Colab (default: '/content/')

    Returns:
    list: List of full paths to the copied files in the Colab environment
    """
    # Mount Google Drive (if not already mounted)
    try:
        drive.mount('/content/drive')
        print("Google Drive mounted successfully.")
    except Exception as e:
        print(f"Drive mounting issue: {e}")
        # Check if drive is already mounted
        if not os.path.exists('/content/drive'):
            raise Exception("Failed to mount Google Drive")

    # Verify the source directory exists
    full_drive_path = drive_folder_path
    if not os.path.exists(full_drive_path):
        raise ValueError(f"Directory not found: {full_drive_path}")

    # Create destination directory if it doesn't exist
    os.makedirs(colab_destination, exist_ok=True)

    # Get all files in the drive directory
    all_files = []
    for item in os.listdir(full_drive_path):
        item_path = os.path.join(full_drive_path, item)
        if os.path.isfile(item_path):
            all_files.append(item_path)

    if not all_files:
        print(f"No files found in {full_drive_path}")
        return []

    # Copy files to Colab directory
    output_files = []
    for source_file in all_files:
        filename = os.path.basename(source_file)
        destination_file = os.path.join(colab_destination, filename)

        try:
            shutil.copy2(source_file, destination_file)
            output_files.append(destination_file)
            print(f"Copied: {filename}")
        except Exception as e:
            print(f"Failed to copy {filename}: {e}")

    print(f"\nSuccessfully copied {len(output_files)} files to {colab_destination}")
    return output_files

output_files = copy_files_from_drive(
    drive_folder_path='/content/drive/MyDrive/WCL2R/PAPER2',
    colab_destination='/content/'
)

print("\nCopied files:")
for file_path in output_files:
    print(f"  - {file_path}")


##Choquet Fuzzy Integral Operator

In [ ]:
# Set consistent data type
DTYPE = tf.float32
NP_DTYPE = np.float32

def extract_significance_from_filename(filename):
    """Extract significance value from filename like 'predictions-0.5190.txt'"""
    try:
        base_name = os.path.basename(filename)
        name_without_ext = os.path.splitext(base_name)[0]
        if '-' in name_without_ext:
            significance_str = name_without_ext.split('-')[-1]
            return float(significance_str)
        else:
            return 0.5
    except (ValueError, IndexError):
        return 0.5

def load_data(output_files, judgements_file):
    """Load ranking function outputs and ground truth judgements"""
    significance_values = [extract_significance_from_filename(f) for f in output_files]
    print(f"Extracted significance values: {significance_values}")

    X_list = []
    for file in output_files:
        # Load scores, handling any non-numeric values
        try:
            scores = np.loadtxt(file, dtype=NP_DTYPE)
            X_list.append(scores)
        except ValueError as e:
            print(f"Error loading {file}: {e}")
            # Try alternative loading method
            scores = []
            with open(file, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            score = float(line)
                            scores.append(score)
                        except ValueError:
                            continue
            X_list.append(np.array(scores, dtype=NP_DTYPE))

    X = np.column_stack(X_list)

    y = []
    with open(judgements_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):  # Skip comment lines
                # Extract the first field (relevance score) before any space
                parts = line.split()
                if parts:  # Make sure line is not empty
                    try:
                        relevance_score = float(parts[0])
                        y.append(relevance_score)
                    except ValueError:
                        # Skip lines that don't start with a numeric relevance score
                        continue

    y = np.array(y, dtype=NP_DTYPE)

    if len(y) != X.shape[0]:
        print(f"Warning: Mismatch in number of documents: {X.shape[0]} in outputs vs {len(y)} in judgements")
        # Trim to the smaller size to avoid errors
        min_size = min(X.shape[0], len(y))
        X = X[:min_size]
        y = y[:min_size]
        print(f"Using first {min_size} documents to match sizes")

    return X, y, significance_values

def init_fuzzy_measure_with_significance(n_sources, significance_values):
    """Initialize fuzzy measure parameters using significance values"""
    n_subsets = 2**n_sources - 1
    measure_params = np.zeros(n_subsets, dtype=NP_DTYPE)

    subsets = get_subsets(n_sources)

    for i, subset in enumerate(subsets):
        if len(subset) == 1:
            ranker_idx = subset[0]
            measure_params[i] = significance_values[ranker_idx]
        else:
            avg_significance = np.mean([significance_values[j] for j in subset])
            measure_params[i] = avg_significance

    # Normalize
    max_param = np.max(measure_params)
    if max_param > 0:
        measure_params = measure_params / (max_param * 1.5)

    return measure_params

def get_subsets(n_sources):
    """Generate all non-empty subsets"""
    all_subsets = []
    for k in range(1, n_sources + 1):
        for subset in itertools.combinations(range(n_sources), k):
            all_subsets.append(subset)
    return all_subsets

def precompute_subset_indices(n_sources):
    """Precompute mapping from subsets to indices for faster lookup"""
    subsets = [()] + get_subsets(n_sources)
    subset_to_idx = {subset: idx for idx, subset in enumerate(subsets)}
    return subset_to_idx

def params_to_measure(measure_params, n_sources):
    """Convert parameters to proper fuzzy measure"""
    subsets = [()] + get_subsets(n_sources)
    all_measures = np.zeros(len(subsets), dtype=NP_DTYPE)

    for i, subset in enumerate(subsets[1:], 1):
        max_subset_measure = 0.0
        for j, smaller_subset in enumerate(subsets[:i]):
            if set(smaller_subset).issubset(set(subset)):
                max_subset_measure = max(max_subset_measure, all_measures[j])

        param_value = tf.nn.softplus(tf.constant(measure_params[i-1], dtype=DTYPE))
        all_measures[i] = max_subset_measure + param_value.numpy()

    full_set_idx = len(subsets) - 1
    normalization_factor = all_measures[full_set_idx]
    if normalization_factor > 0:
        all_measures = all_measures / normalization_factor

    return all_measures

def batch_choquet_integral(X_batch, measure_params, n_sources, subset_to_idx):
    """
    Compute Choquet integral for a batch of documents (MUCH faster)
    """
    batch_size = X_batch.shape[0]
    integrals = np.zeros(batch_size, dtype=NP_DTYPE)

    # Precompute the measure once
    measure = params_to_measure(measure_params, n_sources)

    for b in range(batch_size):
        x = X_batch[b]

        # Sort scores in decreasing order
        sorted_indices = np.argsort(x)[::-1]
        sorted_x = x[sorted_indices]

        # Compute integral
        integral_value = 0.0
        for i in range(n_sources):
            A_i = frozenset(sorted_indices[:i+1])
            subset_idx = subset_to_idx[tuple(sorted(A_i))]
            mu_A = measure[subset_idx]

            if i == n_sources - 1:
                integral_value += sorted_x[i] * mu_A
            else:
                integral_value += (sorted_x[i] - sorted_x[i+1]) * mu_A

        integrals[b] = integral_value

    return integrals

def ranking_loss(y_true, y_pred):
    """Custom ranking loss"""
    y_true = tf.cast(y_true, DTYPE)
    y_pred = tf.cast(y_pred, DTYPE)

    mse_loss = tf.reduce_mean(tf.square(y_true - y_pred))

    # Simplified pairwise loss for speed
    if tf.shape(y_true)[0] > 100:  # Only compute for smaller batches
        return mse_loss

    y_true_exp = tf.expand_dims(y_true, 1)
    y_pred_exp = tf.expand_dims(y_pred, 1)

    true_diff = y_true_exp - tf.transpose(y_true_exp)
    pred_diff = y_pred_exp - tf.transpose(y_pred_exp)

    pair_loss = tf.reduce_mean(tf.nn.relu(-true_diff * pred_diff + 0.1))

    return mse_loss + 0.3 * pair_loss

def learn_fuzzy_measure(X, y, significance_values, n_epochs=200, learning_rate=0.01, batch_size=64, pair_loss_weight=0.3):
    """Learn optimal fuzzy measure with batching for speed"""
    n_docs, n_sources = X.shape

    # Precompute subset indices for faster lookup
    subset_to_idx = precompute_subset_indices(n_sources)

    # Initialize parameters with more randomness
    initial_params = init_fuzzy_measure_with_significance(n_sources, significance_values)
    # Add some randomness to break symmetry
    initial_params = initial_params + np.random.normal(0, 0.1, initial_params.shape)

    measure_params = tf.Variable(initial_params, dtype=DTYPE)

    optimizer = tf.optimizers.Adam(learning_rate=learning_rate)
    history = []

    # Create batches
    indices = np.arange(n_docs)
    n_batches = int(np.ceil(n_docs / batch_size))

    print(f"Training with {n_batches} batches per epoch...")

    for epoch in range(n_epochs):
        epoch_loss = 0
        np.random.shuffle(indices)

        for batch_idx in range(n_batches):
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, n_docs)
            batch_indices = indices[start_idx:end_idx]

            X_batch = X[batch_indices]
            y_batch = y[batch_indices]

            with tf.GradientTape() as tape:
                # Compute predictions for the batch
                y_pred_batch = batch_choquet_integral(
                    X_batch, measure_params, n_sources, subset_to_idx
                )

                y_pred_tensor = tf.constant(y_pred_batch, dtype=DTYPE)
                y_batch_tensor = tf.constant(y_batch, dtype=DTYPE)

                # Use the custom pair_loss_weight parameter
                mse_loss = tf.reduce_mean(tf.square(y_batch_tensor - y_pred_tensor))

                # Simplified pairwise loss for speed
                if tf.shape(y_batch_tensor)[0] <= 100:  # Only compute for smaller batches
                    y_true_exp = tf.expand_dims(y_batch_tensor, 1)
                    y_pred_exp = tf.expand_dims(y_pred_tensor, 1)
                    true_diff = y_true_exp - tf.transpose(y_true_exp)
                    pred_diff = y_pred_exp - tf.transpose(y_pred_exp)
                    pair_loss = tf.reduce_mean(tf.nn.relu(-true_diff * pred_diff + 0.1))
                    loss = mse_loss + pair_loss_weight * pair_loss
                else:
                    loss = mse_loss

            gradients = tape.gradient(loss, [measure_params])
            if gradients[0] is not None:
                optimizer.apply_gradients(zip(gradients, [measure_params]))

            epoch_loss += loss.numpy()

        avg_loss = epoch_loss / n_batches
        history.append(avg_loss)

        if epoch % 20 == 0:
            print(f"Epoch {epoch}, Loss: {avg_loss:.6f}")
            # Print some parameter values to see if they're changing
            if epoch == 0:
                print(f"Initial params range: [{np.min(measure_params.numpy()):.4f}, {np.max(measure_params.numpy()):.4f}]")
            else:
                print(f"Params range: [{np.min(measure_params.numpy()):.4f}, {np.max(measure_params.numpy()):.4f}]")

    return measure_params.numpy(), history

def save_predictions(y_pred, filename="predictions_Choquet_Integral.txt"):
    """Save predictions to file"""
    np.savetxt(filename, y_pred, fmt="%.6f")
    print(f"Predictions saved to {filename}")

def evaluate_predictions(judgements_file, predictions_file):
    """Evaluate predictions using external script and return P@1 score"""
    # Copy evaluation script if needed
    if not os.path.exists("Eval-Score-4.0-WCL2R.pl"):
        # Try to find it in parent directory
        if os.path.exists("/content/drive/MyDrive/Eval-Score-4.0-WCL2R.pl"):
            shutil.copy("/content/drive/MyDrive/Eval-Score-4.0-WCL2R.pl", "/content/")
        else:
            print("Eval-Score-4.0-WCL2R.pl not found. Please ensure it's available.")
            return None, None

    outputFileName = "evaluation_output.txt"

    # Run evaluation
    os.system(f"perl Eval-Score-4.0-WCL2R.pl {judgements_file} {predictions_file} output.txt 0")

    if os.path.exists("output.txt"):
        shutil.copyfile("output.txt", outputFileName)
        with open(outputFileName, "r") as final_output:
            result = final_output.read()

        # Parse P@1 from result
        p_at_1 = parse_p_at_1(result)
        return result, p_at_1
    else:
        print("Evaluation failed - output file not created")
        return None, None

def parse_p_at_1(result_text):
    """Parse P@1 from evaluation result"""
    lines = result_text.split('\n')
    for line in lines:
        if "Average" in line and "P@1" in line:
            parts = line.split()
            # Find the position of P@1 value
            try:
                # The line format is: "Average P@1_value P@2_value ..."
                p_at_1_value = parts[1]  # Second element after "Average"
                return float(p_at_1_value)
            except (ValueError, IndexError):
                continue

    # Alternative parsing if the above doesn't work
    for line in lines:
        if "Average" in line:
            # Try to find the number after P@1
            match = re.search(r'Average\s+([\d.]+)', line)
            if match:
                return float(match.group(1))

    return 0.0

def main(output_files, judgements_file, hyperparams=None):
    """Main function with performance optimizations and hyperparameter support"""
    if hyperparams is None:
        hyperparams = {}

    # Extract hyperparameters or use defaults
    n_epochs = hyperparams.get('n_epochs', 100)
    batch_size = hyperparams.get('batch_size', 32)
    learning_rate = hyperparams.get('learning_rate', 0.01)
    pair_loss_weight = hyperparams.get('pair_loss_weight', 0.3)
    sample_size = hyperparams.get('sample_size', 800)

    print("Loading data...")
    start_time = time.time()
    X, y, significance_values = load_data(output_files, judgements_file)
    print(f"Data loading time: {time.time() - start_time:.2f}s")

    print(f"Loaded {X.shape[0]} documents with {X.shape[1]} ranking functions")

    # Use a smaller subset for faster development
    sample_size = min(sample_size, X.shape[0])
    X_sample, y_sample = X[:sample_size], y[:sample_size]
    print(f"Using sample of {sample_size} documents for faster training")

    X_train, X_val, y_train, y_val = train_test_split(
        X_sample, y_sample, test_size=0.2, random_state=42
    )

    print("Learning fuzzy measure...")
    start_time = time.time()
    optimal_params, history = learn_fuzzy_measure(
        X_train, y_train, significance_values,
        n_epochs=n_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        pair_loss_weight=pair_loss_weight
    )
    print(f"Training time: {time.time() - start_time:.2f}s")

    # Evaluate on validation set
    subset_to_idx = precompute_subset_indices(X_val.shape[1])
    y_val_pred = batch_choquet_integral(X_val, optimal_params, X_val.shape[1], subset_to_idx)
    val_loss = ranking_loss(y_val, y_val_pred).numpy()
    print(f"Validation loss: {val_loss:.6f}")

    # Make final predictions on full dataset
    print("Generating final predictions...")
    y_pred = batch_choquet_integral(X, optimal_params, X.shape[1], subset_to_idx)

    # Save predictions temporarily for evaluation
    temp_pred_file = "temp_predictions.txt"
    save_predictions(y_pred, temp_pred_file)

    # Evaluate and get P@1 score
    eval_result, p_at_1 = evaluate_predictions(judgements_file, temp_pred_file)

    # Clean up temporary file
    if os.path.exists(temp_pred_file):
        os.remove(temp_pred_file)

    return y_pred, optimal_params, val_loss, p_at_1, eval_result

def hyperparameter_tuning(output_files, judgements_file):
    """Perform hyperparameter tuning to find the best configuration based on P@1"""
    # Define hyperparameter grid with more variation
    param_grid = {
        'n_epochs': [100],#, 200],  # Increased epochs
        'batch_size': [32], #, 64],
        'learning_rate': [0.01, 0.02], #, 0.5],  # Higher learning rates
        'pair_loss_weight': [0.1,0.3],#, 0.5, 1.0],  # Wider range
        'sample_size': [800]  # Larger samples [1000,2000]
    }

    best_p_at_1 = 0.0  # Higher P@1 is better
    best_params = {}
    best_predictions = None
    best_measure = None
    best_eval_result = None

    # Generate all combinations of hyperparameters
    keys = param_grid.keys()
    values = param_grid.values()
    total_combinations = np.prod([len(v) for v in values])
    print(f"Testing {total_combinations} hyperparameter combinations...")

    # Iterate through all combinations
    for i, combination in enumerate(itertools.product(*values)):
        hyperparams = dict(zip(keys, combination))
        print(f"\nTesting combination {i+1}/{total_combinations}: {hyperparams}")

        try:
            # Run training with these hyperparameters
            predictions, measure, val_loss, p_at_1, eval_result = main(output_files, judgements_file, hyperparams)

            print(f"P@1 score for this combination: {p_at_1:.4f}")
            print(f"Validation loss: {val_loss:.6f}")

            # Check if this is the best so far
            if p_at_1 > best_p_at_1:
                best_p_at_1 = p_at_1
                best_params = hyperparams
                best_predictions = predictions
                best_measure = measure
                best_eval_result = eval_result
                print(f"New best! P@1: {p_at_1:.4f}")

        except Exception as e:
            print(f"Error with parameters {hyperparams}: {e}")

            traceback.print_exc()
            continue

    print(f"\nBest parameters: {best_params}")
    print(f"Best P@1: {best_p_at_1:.4f}")

    return best_predictions, best_measure, best_params, best_p_at_1, best_eval_result

# Run hyperparameter tuning
print("Starting hyperparameter tuning...")
best_predictions, best_measure, best_params, best_p_at_1, best_eval_result = hyperparameter_tuning(output_files, "judgements.txt")

# Save the best predictions
with open('predictions_Choquet_Integral_best.txt', 'w') as f:
    for f1 in best_predictions:
        f.write(f"  {f1:.7e}  \n")

# Evaluate the best model
print("Evaluating best model...")
if best_eval_result:
    print(f"Result of Aggregation by the Choquet Integral:\n")
    print(best_eval_result)

    # Parse and display metrics
    lines = best_eval_result.split('\n')
    for line in lines:
        if "Average" in line :
            print(line)
else:
    # Run evaluation again if needed
    result, p_at_1 = evaluate_predictions("judgements.txt", "predictions_Choquet_Integral_best.txt")
    if result:
        print(f"Result of Aggregation by the Choquet Integral:\n")
        print(result)

print("Process completed successfully!")